## Setup Cells

In [0]:
%pip install langgraph langchain langchain-openai pydantic

In [0]:
import os
from typing import Literal, List, Dict, TypedDict, Optional, Any
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Config
CATALOG = "deepcatalog"
SCHEMA  = "delta_tables"

# API Key
os.environ["OPENAI_API_KEY"] = dbutils.secrets.get(scope="llm-scope", key="openai-key")

# LLM
llm = ChatOpenAI(model="gpt-5-nano", temperature=0)

# Set Spark context
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

print("✅ Setup done")

In [0]:
class AgentState(TypedDict):
    query:        str
    intent:       Optional[dict]
    schema:       Optional[dict]
    sql:          Optional[str]
    result:       Optional[Any]
    eval_result:  Optional[dict]
    diag_result:  Optional[dict]
    errors:       List[str]
    retry_count:  int
    final:        bool
    needs_human:  bool

In [0]:
class IntentFilter(BaseModel):
    column:   str
    operator: str
    value:    str

class IntentResponse(BaseModel):
    intent_type: Literal["count", "select", "aggregate", "other"] = "select"
    targets:     List[str] = Field(default_factory=list)
    filters:     List[IntentFilter] = Field(default_factory=list)
    group_by:    List[str] = Field(default_factory=list)
    complexity:  Literal["low", "medium", "high"] = "low"

class SchemaMapResponse(BaseModel):
    tables:    List[str] = Field(default_factory=list)
    join_keys: List[List[str]] = Field(default_factory=list)
    columns:   Dict[str, List[str]] = Field(default_factory=dict)

class EvalResponse(BaseModel):
    credible:      bool
    reason:        str
    suggested_fix: Optional[str] = None

class DiagResponse(BaseModel):
    error_type:     Literal["schema", "syntax", "logic", "unknown"] = "unknown"
    fix_suggestion: str = ""
    needs_human:    bool = False

print("✅ Models defined")

In [0]:
SYNTHEA_SCHEMA = {
    "tables": {
        "patients":    ["Id", "BIRTHDATE", "DEATHDATE", "GENDER", "RACE", "CITY", "STATE"],
        "conditions":  ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"],
        "medications": ["START", "STOP", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "REASONCODE"],
        "observations":["DATE", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "VALUE", "UNITS", "TYPE"],
        "encounters":  ["Id", "START", "STOP", "PATIENT", "CODE", "DESCRIPTION", "REASONCODE"],
        "procedures":  ["DATE", "PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION"]
    },
    "join_keys": {
        "conditions":  ["patients.Id", "conditions.PATIENT"],
        "medications": ["patients.Id", "medications.PATIENT"],
        "observations":["patients.Id", "observations.PATIENT"],
        "encounters":  ["patients.Id", "encounters.PATIENT"],
        "procedures":  ["patients.Id", "procedures.PATIENT"]
    }
}

print("✅ Schema reference ready")

## Analyser Agent

In [0]:
ANALYZER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a healthcare data query analyzer working with Synthea data.
Parse the user query and return a JSON object with these exact fields:
- intent_type: one of count, select, aggregate, other
- targets: list of ALL entity types involved. Always include related entities
  (e.g., if query mentions a condition, include both "patients" and "conditions")
- filters: list of objects with column, operator, value
- group_by: list of grouping fields if any
- complexity: low, medium, or high

Return ONLY valid JSON. No explanation, no markdown, no backticks."""),
    ("human", "{query}")
])

def analyzer_node(state: dict) -> dict:
    try:
        chain  = ANALYZER_PROMPT | llm
        result = chain.invoke({"query": state["query"]})
        intent = IntentResponse.model_validate_json(result.content)
        return {**state, "intent": intent.model_dump()}
    except Exception as e:
        return {
            **state,
            "intent": None,
            "errors": state.get("errors", []) + [f"Analyzer error: {str(e)}"]
        }

print("✅ Analyzer Agent ready")

In [0]:
# test_state = {
#     "query": "How many female patients have diabetes?",
#     "errors": []
# }

# result = analyzer_node(test_state)
# print("Intent:", result["intent"])
# print("Errors:", result["errors"])

## Schema Mapper Agent

In [0]:
SCHEMA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a healthcare database schema expert working with Synthea data.
Given the user intent, SELECT ONLY the relevant tables and columns needed to answer the query.
Do NOT return the entire schema.

Return a JSON object with:
- tables: list of ONLY the table names needed for this specific query
- join_keys: list of [left_key, right_key] pairs needed (only if joining tables)
- columns: dict of table -> list of ONLY the columns needed for this specific query

Available schema for reference:
{schema_reference}

Return ONLY valid JSON. No explanation, no markdown, no backticks."""),
    ("human", "Intent: {intent}")
])

def schema_mapper_node(state: dict) -> dict:
    try:
        chain  = SCHEMA_PROMPT | llm
        result = chain.invoke({
            "intent":           str(state["intent"]),
            "schema_reference": str(SYNTHEA_SCHEMA)
        })
        schema = SchemaMapResponse.model_validate_json(result.content)
        return {**state, "schema": schema.model_dump()}
    except Exception as e:
        return {
            **state,
            "schema": None,
            "errors": state.get("errors", []) + [f"Schema mapper error: {str(e)}"]
        }

print("✅ Schema Mapper Agent ready")

In [0]:
# test_state = {
#     "query": "How many female patients have diabetes?",
#     "intent": {
#         "intent_type": "count",
#         "targets": ["patients", "conditions"],
#         "filters": [
#             {"column": "gender", "operator": "=", "value": "female"},
#             {"column": "condition", "operator": "=", "value": "diabetes"}
#         ],
#         "group_by": [],
#         "complexity": "low"
#     },
#     "errors": []
# }

# result = schema_mapper_node(test_state)
# print("Schema:", result["schema"])
# print("Errors:", result["errors"])

## SQL Generator Agent

In [0]:
# Cache all samples once at startup
# Replace build_sample_cache() with this
SAMPLED_VALUES = {
    "patients.GENDER": ["M", "F"],
    "patients.RACE":   ["white", "black", "asian", "hispanic", "native"],
    "patients.STATE":  ["Massachusetts", "California", "New York"],
    "observations.TYPE": ["numeric", "text"],
    "encounters.REASONCODE": ["use LIKE pattern"],
    "conditions.DESCRIPTION": ["use LIKE '%keyword%' pattern"],
    "medications.DESCRIPTION": ["use LIKE '%keyword%' pattern"]
}

print(f"✅ {len(SAMPLED_VALUES)} columns preloaded")

# def build_sample_cache():
#     for table, columns in SYNTHEA_SCHEMA["tables"].items():
#         for column in columns:
#             try:
#                 qualified = f"{CATALOG}.{SCHEMA}.{table}"
#                 rows = spark.sql(f"SELECT DISTINCT {column} FROM {qualified} LIMIT 5").collect()
#                 SAMPLED_VALUES[f"{table}.{column}"] = [row[0] for row in rows]
#             except:
#                 SAMPLED_VALUES[f"{table}.{column}"] = []

# build_sample_cache()
# print(f"✅ Sampled {len(SAMPLED_VALUES)} columns")

SQL_GEN_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a Spark SQL expert working with Synthea healthcare data on Azure Databricks.
Generate a valid Spark SQL query based on the intent and schema provided.

Rules:
- Always use fully qualified table names: deepcatalog.delta_tables.tablename
- Use table aliases: p=patients, c=conditions, m=medications, o=observations, e=encounters
- Use ONLY the actual sampled column values provided for filters (not assumed values)
- For partial text matches use LOWER(column) LIKE LOWER('%keyword%')
- Always qualify column names with table alias
- Return ONLY the SQL query. No explanation, no markdown, no backticks."""),
    ("human", """Intent: {intent}
Schema: {schema}
Sampled column values: {sampled_values}
Previous errors (if any): {errors}""")
])

def sql_generator_node(state: dict) -> dict:
    try:
        schema = state["schema"]

        # Read from cache — no Spark jobs at query time
        sampled = {
            k: v for k, v in SAMPLED_VALUES.items()
            if any(k.startswith(table) for table in schema.get("tables", []))
        }

        temp    = 0.0 if state.get("retry_count", 0) == 0 else 0.3
        gen_llm = ChatOpenAI(model="gpt-4o", temperature=temp)

        chain  = SQL_GEN_PROMPT | gen_llm
        result = chain.invoke({
            "intent":         str(state["intent"]),
            "schema":         str(schema),
            "sampled_values": str(sampled),
            "errors":         state.get("errors", [])
        })

        return {**state, "sql": result.content.strip()}

    except Exception as e:
        return {
            **state,
            "sql":    None,
            "errors": state.get("errors", []) + [f"SQL generator error: {str(e)}"]
        }

In [0]:
# test_state = {
#     "query": "How many female patients have diabetes?",
#     "intent": {
#         "intent_type": "count",
#         "targets": ["patients", "conditions"],
#         "filters": [
#             {"column": "gender", "operator": "=", "value": "female"},
#             {"column": "condition", "operator": "=", "value": "diabetes"}
#         ],
#         "group_by": [],
#         "complexity": "low"
#     },
#     "schema": {
#         "tables": ["patients", "conditions"],
#         "join_keys": [["patients.Id", "conditions.PATIENT"]],
#         "columns": {
#             "patients": ["Id", "GENDER"],
#             "conditions": ["PATIENT", "DESCRIPTION"]
#         }
#     },
#     "errors": [],
#     "retry_count": 0
# }

# result = sql_generator_node(test_state)
# print("SQL:", result["sql"])
# print("Errors:", result["errors"])

## Executor Agent

In [0]:
import re

DESTRUCTIVE = re.compile(
    r'\b(DROP|DELETE|TRUNCATE|INSERT|UPDATE|ALTER|CREATE|REPLACE)\b',
    re.IGNORECASE
)

def executor_node(state: dict) -> dict:
    sql = state.get("sql")

    if not sql:
        return {
            **state,
            "errors": state.get("errors", []) + ["Executor error: no SQL to execute"]
        }

    if DESTRUCTIVE.search(sql):
        return {
            **state,
            "errors": state.get("errors", []) + ["Blocked: destructive SQL detected"],
            "result": None
        }

    try:
        result_df = spark.sql(sql)
        result    = result_df.collect()
        return {**state, "result": result}
    except Exception as e:
        return {
            **state,
            "result": None,
            "errors": state.get("errors", []) + [f"Execution error: {str(e)}"]
        }

print("✅ Executor Agent ready")

## Evaulator Agent

In [0]:
EVAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a healthcare data quality evaluator.
Check if the query result is credible given the original question.
Flag issues like:
- COUNT queries returning multiple rows
- Percentages over 100%
- Empty results where data should exist
- Results that contradict medical logic

Return ONLY valid JSON: {"credible": bool, "reason": str, "suggested_fix": str or null}
No explanation, no markdown, no backticks."""),
    ("human", """Question: {query}
SQL: {sql}
Result (sample): {result}""")
])

def evaluator_node(state: dict) -> dict:
    try:
        result_sample = str(state.get("result", "None"))[:500]
        chain  = EVAL_PROMPT | llm
        result = chain.invoke({
            "query":  state["query"],
            "sql":    state["sql"],
            "result": result_sample
        })
        eval_result = EvalResponse.model_validate_json(result.content)
        is_final    = eval_result.credible and not state.get("errors")
        return {**state, "eval_result": eval_result.model_dump(), "final": is_final}
    except Exception as e:
        return {
            **state,
            "eval_result": None,
            "errors": state.get("errors", []) + [f"Evaluator error: {str(e)}"]
        }

print("✅ Evaluator Agent ready")

## Diagonistican Agent

In [0]:
DIAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a SQL debugging expert for healthcare data on Databricks.
Diagnose the issue and classify it as one of:
- schema: wrong table or column name
- syntax: invalid SQL syntax
- logic: correct SQL but wrong answer
- unknown: cannot determine

Return ONLY valid JSON: {"error_type": str, "fix_suggestion": str, "needs_human": bool}
Set needs_human to true only if the fix requires domain expertise beyond SQL.
No explanation, no markdown, no backticks."""),
    ("human", """Question: {query}
SQL: {sql}
Errors: {errors}
Eval feedback: {eval_feedback}""")
])

def diagnostician_node(state: dict) -> dict:
    try:
        chain  = DIAG_PROMPT | llm
        result = chain.invoke({
            "query":        state["query"],
            "sql":          state.get("sql", ""),
            "errors":       state.get("errors", []),
            "eval_feedback":str(state.get("eval_result", {}))
        })
        diag = DiagResponse.model_validate_json(result.content)
        return {
            **state,
            "diag_result":  diag.model_dump(),
            "needs_human":  diag.needs_human,
            "retry_count":  state.get("retry_count", 0) + 1,
            "errors":       state.get("errors", []) + [f"Fix suggestion: {diag.fix_suggestion}"]
        }
    except Exception as e:
        return {
            **state,
            "diag_result": None,
            "errors": state.get("errors", []) + [f"Diagnostician error: {str(e)}"]
        }

print("✅ Diagnostician Agent ready")

## Human Agent

In [0]:
def human_node(state: dict) -> dict:
    print("=" * 60)
    print("⚠️  HUMAN REVIEW REQUIRED")
    print(f"Query    : {state['query']}")
    print(f"Last SQL : {state.get('sql')}")
    print(f"Errors   : {state.get('errors')}")
    print(f"Diag     : {state.get('diag_result')}")
    print("=" * 60)
    return {**state, "final": True, "needs_human": True}

print("✅ Human Agent ready")

## LangGraph Setup

In [0]:
from langgraph.graph import StateGraph, END

def route_after_executor(state: dict) -> str:
    if state.get("errors") and not state.get("result"):
        return "diagnose"
    return "evaluate"

def route_after_eval(state: dict) -> str:
    if state.get("final"):
        return END
    if state.get("retry_count", 0) >= 3:
        return "human"
    return "diagnose"

def route_after_diag(state: dict) -> str:
    if state.get("needs_human") or state.get("retry_count", 0) >= 3:
        return "human"
    return "generate_sql"

def build_graph():
    graph = StateGraph(AgentState)

    graph.add_node("analyze",      analyzer_node)
    graph.add_node("map_schema",   schema_mapper_node)
    graph.add_node("generate_sql", sql_generator_node)
    graph.add_node("execute",      executor_node)
    graph.add_node("evaluate",     evaluator_node)
    graph.add_node("diagnose",     diagnostician_node)
    graph.add_node("human",        human_node)

    graph.set_entry_point("analyze")
    graph.add_edge("analyze",      "map_schema")
    graph.add_edge("map_schema",   "generate_sql")
    graph.add_edge("generate_sql", "execute")
    graph.add_edge("human",        END)

    graph.add_conditional_edges("execute",  route_after_executor, {"evaluate": "evaluate", "diagnose": "diagnose"})
    graph.add_conditional_edges("evaluate", route_after_eval,     {END: END, "diagnose": "diagnose", "human": "human"})
    graph.add_conditional_edges("diagnose", route_after_diag,     {"generate_sql": "generate_sql", "human": "human"})

    return graph.compile()

app = build_graph()
print("✅ LangGraph pipeline ready")

In [0]:
def run_query(user_question: str):
    initial_state = {
        "query":       user_question,
        "intent":      None,
        "schema":      None,
        "sql":         None,
        "result":      None,
        "eval_result": None,
        "diag_result": None,
        "errors":      [],
        "retry_count": 0,
        "final":       False,
        "needs_human": False
    }

    final_state = app.invoke(initial_state)

    print("\n" + "=" * 60)
    if final_state.get("needs_human"):
        print("⚠️  Escalated to human review")
    elif final_state.get("result"):
        print("✅ Result:")
        for row in final_state["result"]:
            print(row)
        print(f"\nSQL:\n{final_state['sql']}")
    else:
        print("❌ Failed")
        print("Errors:", final_state["errors"])
    print("=" * 60)

    return final_state

## Test

In [0]:
run_query("How many total patients are there")